# Практика 20. Лінійне програмування

**Студент:** Huz_Yurii  
**Група:** IT-41  
**Варіант:** N=3, Одеса

Мета: сформулювати виробничу задачу лінійного програмування, розв'язати її за допомогою `scipy.optimize.linprog`, інтерпретувати оптимальний план і перевірити чутливість до зміни ресурсних лімітів.

## Завдання 1. Математична постановка

Позначимо `x1` — кількість одиниць виробу А, що випускаються за тиждень; `x2` — кількість одиниць виробу Б за тиждень.

Для Одеси прибуток за одиницю виробу А дорівнює 55 грн, виробу Б — 65 грн. Виріб А/Б споживає відповідно 43/32 кг сировини, 35/35 машино-год обладнання та 12/11 кВт·год електроенергії. Тижневі ліміти — 320 кг сировини, 260 машино-год і 110 кВт·год.

```text
максимізувати   P = 55*x1 + 65*x2                         (грн/тиждень)

43*x1 + 32*x2 <= 320                                      (сировина, кг)
35*x1 + 35*x2 <= 260                                      (час обладнання, машино-год)
12*x1 + 11*x2 <= 110                                      (електроенергія, кВт·год)
x1 >= 0, x2 >= 0
```

Кількості тут є неперервними змінними лінійної релаксації; практичне тлумачення дробових обсягів розглянемо після оптимізації.

## Завдання 2. Розв'язання через `linprog`

`linprog` мінімізує цільову функцію, тому для початкової задачі максимізації прибутку передаємо коефіцієнти зі знаком мінус. Нижче перевіряємо `status == 0` перед використанням оптимуму.

In [1]:
import numpy as np
from scipy.optimize import linprog

profit_per_unit = np.array([55, 65], dtype=float)
A_ub = np.array([
    [43, 32],
    [35, 35],
    [12, 11],
], dtype=float)
resource_limits = np.array([320, 260, 110], dtype=float)
resource_names = ["Сировина", "Час обладнання", "Електроенергія"]

res = linprog(
    -profit_per_unit,
    A_ub=A_ub,
    b_ub=resource_limits,
    bounds=[(0, None), (0, None)],
    method="highs",
)
print("res.x =", res.x)
print("res.fun =", res.fun, "(мінімізоване від'ємне значення прибутку)")
print("Справжній максимальний прибуток =", -res.fun, "грн/тиждень")
print("res.status =", res.status)
print("res.message =", res.message)
if res.status != 0:
    raise RuntimeError(f"Оптимізація не завершилася успішно: {res.message}")

res.x = [0.         7.42857143]
res.fun = -482.85714285714283 (мінімізоване від'ємне значення прибутку)
Справжній максимальний прибуток = 482.85714285714283 грн/тиждень
res.status = 0
res.message = Optimization terminated successfully. (HiGHS Status 7: Optimal)


Оптимізатор має повернути `res.status == 0` — «Optimization terminated successfully». Тільки після цієї перевірки інтерпретуємо `res.x`; справжній прибуток дорівнює `-res.fun`, оскільки задача була передана як мінімізація від'ємного прибутку.

## Завдання 3. Інтерпретація оптимального плану

Обчислимо використання кожного ресурсу та виведемо оптимальні кількості й прибуток із точністю, яка не приховує дробовість розв'язку.

In [2]:
x1, x2 = res.x
max_profit = -res.fun
resource_used = A_ub @ res.x
print(f"Виріб А: {x1:.4f} од./тиждень")
print(f"Виріб Б: {x2:.4f} од./тиждень")
print(f"Максимальний прибуток: {max_profit:.2f} грн/тиждень")
print(f"Відношення А/Б: {x1 / x2 if x2 else np.nan:.4f}")
for name, used, limit in zip(resource_names, resource_used, resource_limits):
    print(f"{name}: використано {used:.3f} з {limit:.3f}")

Виріб А: 0.0000 од./тиждень
Виріб Б: 7.4286 од./тиждень
Максимальний прибуток: 482.86 грн/тиждень
Відношення А/Б: 0.0000
Сировина: використано 237.714 з 320.000
Час обладнання: використано 260.000 з 260.000
Електроенергія: використано 81.714 з 110.000


Оптимум неперервної LP-моделі — 0 одиниць виробу А та приблизно **7.4286 одиниці виробу Б** на тиждень, прибуток — приблизно **482.86 грн**. Дробове число є коректним для неперервної математичної моделі (наприклад, якщо продукцію можна планувати партіями/частками). Якщо одиниці виробу неподільні, це вже задача цілочислової оптимізації. Просте округлення до 7 одиниць дасть допустимий план, але не гарантує найкращий цілочисловий план: округлення може порушити обмеження або пропустити кращу комбінацію, тож для гарантованого цілого оптимуму потрібен integer solver.

## Завдання 4. Активні обмеження та запаси

Для `A_ub @ x <= b_ub` slack — невикористана кількість ресурсу. Нульовий slack означає активне обмеження й ресурс-вузьке місце в поточному оптимумі.

In [4]:
slack = res.slack
for name, used, limit, remaining in zip(resource_names, resource_used, resource_limits, slack):
    state = "активне" if np.isclose(remaining, 0, atol=1e-7) else "неактивне"
    print(f"{name}: використано {used:.3f}/{limit:.3f}, запас={remaining:.3f} — {state}")

active_indices = np.flatnonzero(np.isclose(slack, 0, atol=1e-7))
inactive_indices = np.flatnonzero(~np.isclose(slack, 0, atol=1e-7))
active_names = [resource_names[i] for i in active_indices]
inactive_names = [resource_names[i] for i in inactive_indices]
print("Активні ресурси:", active_names)
print("Неактивні ресурси:", inactive_names)

Сировина: використано 237.714/320.000, запас=82.286 — неактивне
Час обладнання: використано 260.000/260.000, запас=0.000 — активне
Електроенергія: використано 81.714/110.000, запас=28.286 — неактивне
Активні ресурси: ['Час обладнання']
Неактивні ресурси: ['Сировина', 'Електроенергія']


У базовому оптимумі активним є **час обладнання**: використано всі 260 машино-год, slack дорівнює 0. Це поточне вузьке місце; саме розширення доступного часу може дозволити збільшити випуск і прибуток. Сировина має запас приблизно **82.286 кг**, а електроенергія — приблизно **28.286 кВт·год**, тому ці два обмеження не є вузькими місцями в знайденому оптимумі.

## Завдання 5. What-if аналіз

Збільшуємо активний ліміт часу обладнання на 15%, а потім ліміт одного неактивного ресурсу — сировини — на 50%. В обох випадках повторно розв'язуємо ту саму задачу з незмінними прибутками й нормами витрат.

In [3]:
def solve_with_limits(limits):
    return linprog(
        -profit_per_unit,
        A_ub=A_ub,
        b_ub=limits,
        bounds=[(0, None), (0, None)],
        method="highs",
    )

time_index = resource_names.index("Час обладнання")
time_limits = resource_limits.copy()
time_limits[time_index] *= 1.15
res_active_15 = solve_with_limits(time_limits)
if res_active_15.status != 0:
    raise RuntimeError(f"What-if для активного ресурсу не вдалося: {res_active_15.message}")

raw_index = resource_names.index("Сировина")
raw_limits = resource_limits.copy()
raw_limits[raw_index] *= 1.50
res_inactive_50 = solve_with_limits(raw_limits)
if res_inactive_50.status != 0:
    raise RuntimeError(f"What-if для неактивного ресурсу не вдалося: {res_inactive_50.message}")

scenarios = [
    ("Базовий план", res),
    ("Час +15%", res_active_15),
    ("Сировина +50%", res_inactive_50),
]
for label, result in scenarios:
    quantities = result.x
    print(
        f"{label}: A={quantities[0]:.4f}, B={quantities[1]:.4f}, "
        f"прибуток={-result.fun:.2f} грн/тиждень, slack={np.round(result.slack, 3)}"
    )
print(f"Зміна прибутку після збільшення часу: {-res_active_15.fun - max_profit:.2f} грн/тиждень")
print(f"Зміна прибутку після збільшення сировини: {-res_inactive_50.fun - max_profit:.2f} грн/тиждень")

Базовий план: A=0.0000, B=7.4286, прибуток=482.86 грн/тиждень, slack=[82.286  0.    28.286]
Час +15%: A=0.0000, B=8.5429, прибуток=555.29 грн/тиждень, slack=[46.629  0.    16.029]
Сировина +50%: A=0.0000, B=7.4286, прибуток=482.86 грн/тиждень, slack=[242.286   0.     28.286]
Зміна прибутку після збільшення часу: 72.43 грн/тиждень
Зміна прибутку після збільшення сировини: 0.00 грн/тиждень


Збільшення активного ресурсу — часу обладнання — на 15% підвищує його ліміт із 260 до 299 машино-год. Оптимальний випуск виробу Б збільшується приблизно з **7.429** до **8.543** одиниці, а прибуток — із **482.86** до **555.29 грн/тиждень**; структура випуску залишається лише з виробу Б (x1=0). Це підтверджує, що додатковий вузький ресурс має цінність.

Збільшення неактивного ресурсу — сировини — на 50% не змінює оптимальний план і прибуток: час обладнання все одно обмежує випуск, а сировина вже мала запас. Це узгоджується з ідеєю, що додатковий ресурс не підвищує прибуток, доки він не є вузьким місцем.

## Контрольні питання

1. **Чому `linprog` мінімізує?** Це формат задачі, який реалізує функція. Щоб максимізувати прибуток `p·x`, передаємо ціль `-p·x`; мінімальний результат буде `res.fun = -Pmax`, отже справжній максимум дорівнює `-res.fun`.
2. **Чому `>=` множимо на −1?** `linprog` приймає ресурсні нерівності як `A_ub @ x <= b_ub`. Домноження обох частин `a·x >= L` на −1 змінює напрям нерівності й дає `-a·x <= -L`, еквівалентну форму, сумісну з `A_ub`, `b_ub`.
3. **Активне й неактивне обмеження.** Активне має нульовий slack: весь ресурс використаний у оптимумі й може обмежувати додатковий прибуток. Неактивне має запас; збільшення такого ліміту зазвичай не змінює оптимум, як у сценарії сировини +50%.
4. **Чи узгоджується what-if?** Так: збільшення активного часу обладнання на 15% підвищило прибуток приблизно на 72.43 грн/тиждень, тоді як збільшення неактивної сировини на 50% не дало приросту прибутку. Це відповідає висновку про час як вузьке місце базового плану.

## Підсумок

Для варіанта N=3 математично сформульовано й розв'язано задачу виробничого планування Одеси. Визначено оптимальний план і прибуток, активний ресурс часу обладнання та запаси двох інших ресурсів; what-if перевірка показала зростання прибутку при розширенні активного ресурсу й відсутність ефекту від збільшення неактивного.